In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, VGG16, MobileNetV3Large
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from datetime import datetime
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

   
   #

In [ ]:
# =================================================================
# PHASE 1 & 2 : BUSINESS & DATA UNDERSTANDING
# Objectif : Classifier 4 types d'états cérébraux avec haute précision.
# Données : Dataset Masoud Nickparvar (~7000 images IRM).
# =================================================================
path = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
train_dir = os.path.join(path, 'Training')
test_dir = os.path.join(path, 'Testing')

#

In [ ]:


# =================================================================
# FONCTION DE DATA MINING : AMÉLIORATION DU CONTRASTE (CLAHE)
# Essentiel pour réduire les erreurs sur les petits méningiomes.
# =================================================================
def enhance_contrast(image):
    # Conversion en uint8 pour OpenCV
    img = np.array(image, dtype=np.uint8)

    # Application de CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    for i in range(3):
        img[:,:,i] = clahe.apply(img[:,:,i])

    # Retourne l'image normalisée pour EfficientNet
    return preprocess_input(img.astype(np.float32))

# =================================================================
# PHASE 3 : DATA PREPARATION
# =================================================================

# On augmente légèrement les paramètres de transformation pour
# générer assez de variantes uniques pour la classe "no_tumor".
train_datagen = ImageDataGenerator(
    preprocessing_function=enhance_contrast,
    rotation_range=20,       # Augmenté (était 15) pour plus de diversité
    width_shift_range=0.15,  # Augmenté pour varier la position
    height_shift_range=0.15,
    zoom_range=0.25,         # Ajouté : Crucial pour simuler des tailles différentes
    shear_range=0.1,         # Ajouté : Pour varier l'angle de vue
    horizontal_flip=True,
    fill_mode='reflect',     # Meilleur rendu pour les bords d'IRM
    validation_split=0.2
)

# Le générateur de test reste sobre mais doit inclure le contraste
test_datagen = ImageDataGenerator(preprocessing_function=enhance_contrast)

# Chargement des données
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)


#

In [ ]:
# =================================================================
# PHASE 4 : MODELING (Extraction de motifs profonds)
# Architecture : EfficientNetB0 + Transfer Learning.
# =================================================================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True # Fine-tuning total

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x) # Réduction du décalage de covariance interne
x = Dropout(0.4)(x) # Régularisation contre l'overfitting
outputs = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Entraînement
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

print("\n--- Entraînement en cours (Phase Modeling) ---")
start_time = datetime.now()
history = model.fit(train_gen, epochs=20, validation_data=val_gen, callbacks=callbacks)
end_time = datetime.now()
training_duration = end_time - start_time
# Formatage lisible HH:MM:SS
total_seconds = int(training_duration.total_seconds())
hours = total_seconds // 3600
minutes = (total_seconds % 3600) // 60
seconds = total_seconds % 60
print(f"Temps d'entraînement : {hours:d}h {minutes:02d}m {seconds:02d}s")

#

In [ ]:
# =================================================================
# PHASE 5 : EVALUATION (Validation des connaissances extraites)
# Analyse des métriques de performance : Précision, Rappel, F1-Score.
# =================================================================
y_pred_prob = model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_gen.classes
categories = list(train_gen.class_indices.keys())

# Calcul des métriques avancées
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
acc = accuracy_score(y_true, y_pred)

print(f"\n--- RÉSULTATS DE L'ÉVALUATION ---")
print(f"Accuracy Globale : {acc:.2%}") #
print(f"Précision (Moyenne) : {precision:.2%}")
print(f"Rappel (Recall Moyenne) : {recall:.2%}") # Crucial en médical
print(f"F1-Score (Moyenne) : {f1:.2%}")

print("\n--- RAPPORT DÉTAILLÉ PAR CLASSE ---")
print(classification_report(y_true, y_pred, target_names=categories))

# Matrice de confusion (Visual Data Mining)
print("\n--- Matrice de Confusion ---")
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=categories, yticklabels=categories)
plt.title('Matrice de Confusion Finale (Phase Evaluation)')
plt.show()


# Afficher les courbes d'apprentissage (accuracy / loss)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history.get('accuracy', []), label='Train')
plt.plot(history.history.get('val_accuracy', []), label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history.get('loss', []), label='Train')
plt.plot(history.history.get('val_loss', []), label='Val')
plt.title('Loss')
plt.legend()
plt.show()

#

In [ ]:
# =================================================================
# PHASE 6 : DEPLOYMENT
# Exportation pour intégration Flask.
# =================================================================

model_filename = f'efficientnet_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.h5'
model.save(model_filename)
print("\nModèle sauvegardé : Prêt pour le déploiement Flask.")